# Conversion for Mobile Deployment


- PyTorch → ONNX
- ONNX → TensorFlow
- TensorFlow → TFLite + INT8 Quantization

References
- https://github.com/DanieliusKr/onnx-example/blob/main/onnx_example.ipynb
- https://www.geeksforgeeks.org/machine-learning/convert-pytorch-model-to-tf-lite-with-onnx-tf/

## Conversion to ONNX

### Import

In [ ]:
!pip install onnx
!pip install onnxruntime
!pip install onnx-tf
!pip install tensorflow-addons
!pip install tensorflow-probability

In [ ]:
import torch as torch
import torch.nn as nn
import time
import os

import torch.onnx
import onnxruntime as ort
from onnx_tf.backend import prepare
import tensorflow as tf

### Model

In [ ]:
# Load the trained model


# Input for inference
input = torch.tensor([[1.0, 2.0]])

# Perform inference
output = model(input)
print("PyTorch Inference Output:", output.detach().numpy())

### Conversion to ONNX

In [ ]:
onnx_path = "convlstm.onnx"

# Convert from PyTorch to ONNX
torch.onnx.export(
    model,                         
    input,                        
    onnx_path,                      
    input_names=["input"],          
    output_names=["output"],        
)

print(f"Success: conversion to ONNX. File is saved at: {onnx_path}")

### Loads ONNX

In [ ]:
# Load the ONNX model
onnx_path = "convlstm.onnx"
ort_session = ort.InferenceSession(onnx_path)

# Prepare sample input data (same shape as the PyTorch model)
# onnx_input

# Run inference on the ONNX model
onnx_output = ort_session.run(None, {"input": onnx_input})
print(f"ONNX Inference Output: {onnx_output}")

### Comparison of Speed

In [ ]:
# PyTorch inference time
total_time = 0
iterations = 50

for i in range(iterations):
  # input = np.random.rand(1, 2)
  start_time = time.time()
  output = model(input)
  end_time = time.time()
  total_time += (end_time - start_time)

print(f"Total time: {round(total_time/iterations, 2)} seconds")

# ONNX inference time
total_time = 0
iterations = 50
onnx_path = "convlstm.onnx"
ort_session = ort.InferenceSession(onnx_path)

for i in range(iterations):
  # input = np.random.rand(1, 2).astype(np.float32)
  start_time = time.time()
  output = ort_session.run(None, {"input": input})
  end_time = time.time()
  total_time += (end_time - start_time)

print(f"Total time: {round(total_time/iterations, 2)} seconds")

## Conversion to TensorFlow

In [ ]:
onnx_model = onnx.load(onnx_path)
tf_model = prepare(onnx_model)

tf_dir = "tf_model"
os.makedirs(tf_dir, exist_ok=True)

# Convert from ONNX to Tensorflow
tf_model.export_graph(tf_dir)
print(f"Success: conversion to Tensorflow. File is saved at: {tf_dir}")

## Conversion to TFLite

In [ ]:
# Convert from Tensorflow to TFLite
converter = tf.lite.TFLiteConverter.from_saved_model(tf_dir)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

# Save the TFLite model
with open ("convlstm.tflite", "wb") as f:
    f.write(tflite_model)

print(f"Success: conversion to TFLite. File is saved at: {os.path.join(tf_dir, "convlstm.tflite")}")